In [57]:
import random
import numpy as np
import math
import statistics
from collections import defaultdict

# ==================
# BASE CASE - ROTTERDAM
# ==================
GRID_SIZE = 100                   # 100x100 → 10.000 agents (elk ±68 personen)
NUM_AGENTS = GRID_SIZE ** 2
SIMULATION_MONTHS = 12
GEWICHT_PER_STUK = 0.24
INIT_RECYCLING_PERCENTAGE = 0.5   # 50% basis recycling
AFGEDANKT_PER_JAAR = 50
AFGEDANKT_PER_MAAND = AFGEDANKT_PER_JAAR / 12

# ==================
# HUIDIGE ROTTERDAM-SITUATIE
# ==================
# Beleidsvormen: GEEN beloning, huidige infrastructuur
beloning_per_stuk = 0.0           # geen beloning
beloning_groeisnelheid = 0.8
beloning_midden = 3.0   

aantal_inzamelpunten = 4        # huidige inzamelbakken Rotterdam
max_effect_gemak = 0.15
gemak_middelpunt = 4
gemak_scherpte = 0.6
afstand_invloed = 1.5

# ==================
# HULPFUNCTIES
# ==================
def logistiek_effect(x, max_effect, m, k):
    return max_effect / (1 + math.exp(-k * (x - m)))

def logistiek_beloning(beloning, groeisnelheid=0.8, midden=5):
    return 1 / (1 + math.exp(-groeisnelheid * (beloning - midden)))

def genereer_inzamelpunten(aantal, grid_size):
    locaties = set()
    while len(locaties) < aantal:
        locaties.add((random.randint(0, grid_size - 1), random.randint(0, grid_size - 1)))
    return list(locaties)

def afstand(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def afstand_tot_dichtstbijzijnde_inzamelpunt(agent_loc, inzamelpunten):
    return min(afstand(agent_loc, p) for p in inzamelpunten)

def maak_grid_index(agents):
    from collections import defaultdict
    grid_index = defaultdict(list)
    for a in agents:
        grid_index[a.locatie].append(a)
    return grid_index

def buren_recycling_ratio(agent, grid_index):
    x, y = agent.locatie
    buren = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue
            for b in grid_index.get((x + dx, y + dy), []):
                buren.append(b)
    if not buren:
        return 0
    recyclende_buren = [b for b in buren if b.aantal_gerecycled > b.aantal_weggegooid]
    return len(recyclende_buren) / len(buren)

# ==================
# AGENT CLASS
# ==================
class Consument:
    def __init__(self, id, locatie):
        self.id = id
        self.locatie = locatie
        self.milieubewustzijn = random.uniform(0.25, 0.6)
        self.prijsgevoeligheid = random.uniform(0.2, 0.8)
        self.gemaksgevoeligheid = random.uniform(0.3, 0.7)
        self.sociale_druk_gevoeligheid = random.uniform(0.05, 0.4)
        self.sociale_druk = 0
        self.afgedankt_textiel = AFGEDANKT_PER_MAAND
        self.aantal_gerecycled = 0
        self.aantal_weggegooid = 0
        self.afstand_inzamelpunt = None

    def bepaal_recycling_kans(self, basis=INIT_RECYCLING_PERCENTAGE):
        # Beloningseffect is nul bij €0
        beloning_factor = logistiek_beloning(beloning_per_stuk, beloning_groeisnelheid, beloning_midden)
        reward_effect = self.prijsgevoeligheid * 0.3 * beloning_factor

        # Inzamelgemak → logistisch effect van 237 bakken (huidige infrastructuur)
        gemak_effect = logistiek_effect(aantal_inzamelpunten, max_effect_gemak, gemak_middelpunt, gemak_scherpte)

        # Persoonlijke afstand: verder = iets minder gemak
        if self.afstand_inzamelpunt is not None:
            afstand_factor = min(1, self.afstand_inzamelpunt / (GRID_SIZE / 2))**2
            afstand_penalty = self.gemaksgevoeligheid * afstand_invloed * afstand_factor
        else:
            afstand_penalty = 0
        gemak_effect_adjusted = max(0, gemak_effect - afstand_penalty)

        sociaal_effect = 0.3 * self.sociale_druk_gevoeligheid * self.sociale_druk

        totale_effecten = max(0, reward_effect) + max(0, gemak_effect_adjusted) + max(0, sociaal_effect)
        bias_correctie = 0.05   # omdat je nu gemiddeld 55% krijgt i.p.v. 50%
        recycling_kans = min(1, max(0, basis + totale_effecten - bias_correctie))

        return recycling_kans

    def update_gedrag(self):
        p_recycle = self.bepaal_recycling_kans()
        gerecycled, weggegooid = 0, 0
        for _ in range(int(self.afgedankt_textiel)):
            if random.random() < p_recycle:
                gerecycled += 1
            else:
                weggegooid += 1
        self.aantal_gerecycled += gerecycled
        self.aantal_weggegooid += weggegooid

# ==================
# MODEL INITIALISATIE
# ==================
inzamelpunten = genereer_inzamelpunten(aantal_inzamelpunten, GRID_SIZE)
agents = [Consument(id=i, locatie=(i // GRID_SIZE, i % GRID_SIZE)) for i in range(NUM_AGENTS)]
for a in agents:
    a.afstand_inzamelpunt = afstand_tot_dichtstbijzijnde_inzamelpunt(a.locatie, inzamelpunten)

# ==================
# SIMULATIE
# ==================
for maand in range(1, SIMULATION_MONTHS + 1):
    grid_index = maak_grid_index(agents)
    for a in agents:
        a.sociale_druk = buren_recycling_ratio(a, grid_index)
    for a in agents:
        a.update_gedrag()

# ==================
# RESULTATEN
# ==================
totaal_gerecycled = sum(a.aantal_gerecycled for a in agents)
totaal_weggegooid = sum(a.aantal_weggegooid for a in agents)
recycling_percentage = totaal_gerecycled / (totaal_gerecycled + totaal_weggegooid)

totaal_tons_recycled = (totaal_gerecycled * GEWICHT_PER_STUK) / 1000
totaal_tons_restafval = (totaal_weggegooid * GEWICHT_PER_STUK) / 1000
gemiddelde_afstand = statistics.mean(a.afstand_inzamelpunt for a in agents)
afstand_per_agent = [a.afstand_inzamelpunt for a in agents]
recycling_per_agent = [a.aantal_gerecycled / (a.aantal_gerecycled + a.aantal_weggegooid) for a in agents]
correlatie = np.corrcoef(afstand_per_agent, recycling_per_agent)[0, 1]

# ==================
# OUTPUT
# ==================
print("=== BASE CASE: ROTTERDAM ===")
print(f"Inwoners (gesimuleerd): {NUM_AGENTS} agents (~{675000 / NUM_AGENTS:.1f} personen elk)")
print(f"Aantal ingezamelpunten: {aantal_inzamelpunten}")
print(f"Beloning per kledingstuk: €{beloning_per_stuk:.2f} (geen beloning)")
print(f"Verondersteld basisgedrag: {INIT_RECYCLING_PERCENTAGE*100:.0f}% gerecycled")
print("\n=== RESULTATEN ===")
print(f"Totaal gerecycled: {totaal_gerecycled:,} kledingstukken ({totaal_tons_recycled:.2f} ton)")
print(f"Totaal restafval : {totaal_weggegooid:,} kledingstukken ({totaal_tons_restafval:.2f} ton)")
print(f"Recyclingpercentage (model): {recycling_percentage:.2%}")
print(f"Gemiddelde afstand tot inzamelpunt (grid‑eenheden): {gemiddelde_afstand:.2f}")
print(f"Correlatie afstand ↔ recycling: {correlatie:.3f}")
print("\n=== Conclusie ===")
print("Dit is de nulmeting: huidige situatie zonder beleidsmaatregelen, "
      "50% recycling, 237 bakken en geen financiële stimulans.")

=== BASE CASE: ROTTERDAM ===
Inwoners (gesimuleerd): 10000 agents (~67.5 personen elk)
Aantal ingezamelpunten: 4
Beloning per kledingstuk: €0.00 (geen beloning)
Verondersteld basisgedrag: 50% gerecycled

=== RESULTATEN ===
Totaal gerecycled: 238,375 kledingstukken (57.21 ton)
Totaal restafval : 241,625 kledingstukken (57.99 ton)
Recyclingpercentage (model): 49.66%
Gemiddelde afstand tot inzamelpunt (grid‑eenheden): 24.82
Correlatie afstand ↔ recycling: -0.255

=== Conclusie ===
Dit is de nulmeting: huidige situatie zonder beleidsmaatregelen, 50% recycling, 237 bakken en geen financiële stimulans.
